<a href="https://colab.research.google.com/github/erikromerovelasco-dev/pandas-/blob/main/PROYECTO_3_CLASIFICACI%C3%93N_DE_CORREOS_ELECTR%C3%93NICOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PROYECTO 3. CLASIFICACIÓN DE CORREOS ELECTRÓNICOS

In [11]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# Cargar dataset
# Read the file as a single column first, then parse manually with regex
df_temp = pd.read_csv("correos.csv", header=None, encoding='latin1')

# Define a regex pattern to extract ID, Correo (potentially quoted), and Clase
# Example line structure: M01,"email content",spam
pattern = r'^([^,]+),"([^"]*)",(.*)$'

# Apply the regex to the single column and assign to new columns
df_parsed = df_temp[0].str.extract(pattern)
df_parsed.columns = ["ID", "Correo", "Clase"]

# The actual data starts from the second row (index 1) after the header
df = df_parsed.iloc[1:].copy()

# Limpieza
def limpiar(texto):
    # Ensure 'texto' is a string before applying .lower()
    if isinstance(texto, str):
        texto = texto.lower()
        texto = re.sub(r'[^a-záéíóúñ ]', '', texto)
    else:
        texto = '' # Replace non-string (e.g., NaN) with empty string
    return texto

df["Correo"] = df["Correo"].apply(limpiar)

# TF-IDF
vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(df["Correo"])

y = df["Clase"]

# División
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Modelo
modelo = MultinomialNB()
modelo.fit(X_train, y_train)

# Predicciones
predicciones = modelo.predict(X_test)

# Resultados
print("Matriz de Confusión")
print(confusion_matrix(y_test, predicciones))

print("\nReporte de Clasificación")
print(classification_report(y_test, predicciones))

Matriz de Confusión
[[2 0]
 [0 3]]

Reporte de Clasificación
              precision    recall  f1-score   support

     no_spam       1.00      1.00      1.00         2
        spam       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



b) Cantidad de ejemplos por clase
Clase	Cantidad
spam	12
no_spam	12
Total	24

c) Palabras frecuentes por categoría
Spam
Palabra	Frecuencia
para	4
gana	3
dinero	3
oferta	3
premio	3
gratis	3
No Spam
Palabra	Frecuencia
de	10
la	4
reunión	3
reporte	3
del	3
proyecto	2

Análisis:
Los correos spam contienen términos relacionados con promociones, premios y dinero. Los correos legítimos contienen vocabulario asociado con actividades académicas y laborales.

2. Preparación de los Datos
a) Limpieza del texto

Se realizaron las siguientes acciones:

Conversión a minúsculas.
Eliminación de signos de puntuación.
Eliminación de caracteres especiales.
Eliminación de espacios innecesarios.
b) Representación de los correos

Se utilizó TF-IDF (Term Frequency - Inverse Document Frequency).

Justificación

TF-IDF permite:

Convertir texto a valores numéricos.
Identificar palabras importantes dentro de cada correo.
Reducir el impacto de palabras muy comunes.
Funciona muy bien en tareas de clasificación de texto.



3. Selección del Método de Clasificación
Algoritmo elegido
Naive Bayes Multinomial
Justificación

Este algoritmo es ampliamente utilizado para:

Clasificación de correos electrónicos.
Detección de spam.
Análisis de texto.

Ventajas:

Fácil de implementar.
Rápido de entrenar.
Excelente desempeño en conjuntos de texto pequeños.
Funciona bien con TF-IDF.

4. Entrenamiento del Modelo
División de datos
80% entrenamiento
20% prueba

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
# Entrenamiento
from sklearn.naive_bayes import MultinomialNB

modelo = MultinomialNB()
modelo.fit(X_train, y_train)

MultinomialNB()

5. Evaluación del Desempeño
a) Métricas

In [ ]:
from sklearn.metrics import classification_report

predicciones = modelo.predict(X_test)

print(classification_report(y_test, predicciones))

Métricas utilizadas
Accuracy
Precision
Recall
F1-Score

Debido a que el conjunto de datos es pequeño y las palabras de cada categoría son muy distintivas, es probable obtener una precisión cercana al 100%.


b) Análisis de errores

Los posibles errores ocurren cuando:

Un correo contiene palabras tanto de spam como de no spam.
Aparecen términos nuevos que el modelo nunca observó durante el entrenamiento.

Ejemplo:

Adjunto información sobre una oferta académica.

La palabra "oferta" podría generar confusión.
c) Palabras que influyen en la clasificación
Indicadores de Spam
gana
dinero
oferta
premio
gratis
descuento
Indicadores de No Spam
reunión
reporte
proyecto
tarea
coordinación
revisión
6. Interpretación de Resultados
a) ¿El modelo logró distinguir correctamente entre spam y no spam?

Sí.

Las palabras utilizadas en cada categoría son bastante diferentes, por lo que el modelo puede identificar patrones claros y clasificar correctamente la mayoría de los correos.

b) Relación con las características del dataset

El dataset contiene vocabulario muy específico para cada clase:

Spam → promociones, premios y dinero.
No spam → actividades académicas y administrativas.

Esto facilita la tarea de clasificación.

Conclusiones

¿El modelo logró separar adecuadamente los correos spam de los no spam?
Sí. El modelo identificó correctamente los patrones lingüísticos presentes en cada categoría.

¿Qué características del texto fueron más útiles?
Palabras como "gana", "premio", "gratis" y "oferta" para spam, mientras que "reunión", "reporte" y "proyecto" fueron relevantes para no spam.

¿Qué limitaciones tiene el dataset?
Solo contiene 24 registros.
Poco vocabulario.
No representa todos los tipos reales de spam.

¿Qué mejoras se podrían realizar con más datos?
Incorporar miles de correos reales.
Utilizar eliminación de stopwords.
Aplicar técnicas de NLP más avanzadas.
Probar algoritmos como Random Forest, SVM o redes neuronales.